# 02c DFSI Target Column Audit

This notebook audits the target columns in `DFSI.csv` before any sensitivity analysis. It distinguishes stored DFSI scores, possible rank-like targets, unnamed/index columns, and district/state identifiers, then recomputes the requested reconstructed DFSI using population without `+1`.

## Setup

In [1]:
from pathlib import Path
import re

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 160)
pd.set_option("display.max_colwidth", 180)

EXPECTED_RAW_FILES = [
    "DFSI.csv",
    "District_FloodedArea.csv",
    "District_FloodImpact.csv",
    "India_Flood_Inventory_v3.csv",
]


def find_project_root(start_path):
    start_path = Path(start_path).resolve()
    for candidate in [start_path, *start_path.parents]:
        raw_dir = candidate / "data" / "raw"
        if all((raw_dir / file_name).exists() for file_name in EXPECTED_RAW_FILES):
            return candidate
    raise FileNotFoundError("Could not find data/raw with the expected source CSV files.")


PROJECT_ROOT = find_project_root(Path.cwd())
RAW_DIR = PROJECT_ROOT / "data" / "raw"
TABLE_DIR = PROJECT_ROOT / "outputs" / "tables"
TABLE_DIR.mkdir(parents=True, exist_ok=True)

PATHS = {
    "dfsi": RAW_DIR / "DFSI.csv",
    "flooded_area": RAW_DIR / "District_FloodedArea.csv",
    "flood_impact": RAW_DIR / "District_FloodImpact.csv",
    "flood_inventory": RAW_DIR / "India_Flood_Inventory_v3.csv",
}

print(f"Project root: {PROJECT_ROOT}")

Project root: /Users/abhijithshaji/Documents/india-flood-index-robustness


## Load DFSI and Print Stored Columns

In [2]:
dfsi = pd.read_csv(PATHS["dfsi"], low_memory=False)
flooded_area = pd.read_csv(PATHS["flooded_area"], low_memory=False)
flood_impact = pd.read_csv(PATHS["flood_impact"], low_memory=False)
flood_inventory = pd.read_csv(PATHS["flood_inventory"], low_memory=False)

print("DFSI.csv column names exactly as stored:")
for column in dfsi.columns:
    print(repr(column))

print("\nDFSI.csv preview:")
display(dfsi.head(20))

DFSI.csv column names exactly as stored:
'Unnamed: 0'
'State_Name'
'DFSI'

DFSI.csv preview:


,Unnamed: 0,State_Name,DFSI
0,Patna,BIHAR,19.300564
1,Murshidabad,WEST BENGAL,18.910884
2,Thane,MAHARASHTRA,18.861789
3,Guntur,ANDHRA PRADESH,18.832069
4,North Twenty Four Pargan*,WEST BENGAL,18.770538
5,Nagpur,MAHARASHTRA,18.687114
6,Purba Medinipur,WEST BENGAL,18.370426
7,Lakhimpur,ASSAM,18.337466
8,Gorakhpur,UTTAR PRADESH,18.302576
9,Purba Champaran,BIHAR,18.295788


## Numeric Target Column Audit

Every numeric column in `DFSI.csv` is summarized and classified as rank-like, score-like, code-like, or measurement-like.

In [3]:
def classify_numeric_column(series, column_name):
    non_missing = series.dropna()
    unique_count = int(non_missing.nunique())
    minimum = non_missing.min() if len(non_missing) else np.nan
    maximum = non_missing.max() if len(non_missing) else np.nan
    is_integer_like = bool(np.isclose(non_missing, np.round(non_missing)).all()) if len(non_missing) else False
    monotonic_decreasing = bool(non_missing.is_monotonic_decreasing) if len(non_missing) else False
    monotonic_increasing = bool(non_missing.is_monotonic_increasing) if len(non_missing) else False

    lower_name = column_name.casefold()
    if "rank" in lower_name or (is_integer_like and minimum == 1 and maximum <= len(series) and unique_count > len(series) * 0.8):
        resemblance = "rank"
    elif "code" in lower_name or "id" in lower_name:
        resemblance = "code"
    elif "dfsi" in lower_name or (minimum > 0 and maximum < 100 and unique_count > 20):
        resemblance = "score"
    else:
        resemblance = "measurement"

    return {
        "column_name": column_name,
        "dtype": str(series.dtype),
        "minimum": minimum,
        "maximum": maximum,
        "mean": non_missing.mean() if len(non_missing) else np.nan,
        "median": non_missing.median() if len(non_missing) else np.nan,
        "missing_count": int(series.isna().sum()),
        "unique_values": unique_count,
        "integer_like": is_integer_like,
        "monotonic_increasing": monotonic_increasing,
        "monotonic_decreasing": monotonic_decreasing,
        "resembles": resemblance,
    }

numeric_columns = []
for column in dfsi.columns:
    converted = pd.to_numeric(dfsi[column], errors="coerce")
    if converted.notna().any():
        numeric_columns.append(column)

dfsi_target_column_audit = pd.DataFrame([
    classify_numeric_column(pd.to_numeric(dfsi[column], errors="coerce"), column)
    for column in numeric_columns
])

stored_columns = pd.DataFrame({
    "column_name": dfsi.columns,
    "stored_dtype": [str(dfsi[column].dtype) for column in dfsi.columns],
    "role": [
        "index_or_unnamed_csv_column; district identifier values" if column.casefold().startswith("unnamed") else
        "state identifier column" if column == "State_Name" else
        "published DFSI score column" if column == "DFSI" else
        "unclassified"
        for column in dfsi.columns
    ],
})

dfsi_target_column_audit = stored_columns.merge(dfsi_target_column_audit, on="column_name", how="left")
dfsi_target_column_audit["identified_as_published_dfsi_score"] = dfsi_target_column_audit["column_name"].eq("DFSI")
dfsi_target_column_audit["identified_as_published_dfsi_rank"] = False
dfsi_target_column_audit["identified_as_index_or_unnamed"] = dfsi_target_column_audit["column_name"].str.casefold().str.startswith("unnamed")
dfsi_target_column_audit["identified_as_district_identifier"] = dfsi_target_column_audit["column_name"].eq("Unnamed: 0")
dfsi_target_column_audit["identified_as_state_identifier"] = dfsi_target_column_audit["column_name"].eq("State_Name")
dfsi_target_column_audit["audit_note"] = np.where(
    dfsi_target_column_audit["column_name"].eq("DFSI"),
    "Only stored numeric target-like column; values resemble continuous DFSI scores, not ranks.",
    np.where(
        dfsi_target_column_audit["column_name"].eq("Unnamed: 0"),
        "Unnamed CSV column stores district names, not a numeric rank.",
        np.where(
            dfsi_target_column_audit["column_name"].eq("State_Name"),
            "State identifier column.",
            "No special target role identified."
        )
    )
)

display(dfsi_target_column_audit)
dfsi_target_column_audit.to_csv(TABLE_DIR / "dfsi_target_column_audit.csv", index=False)

published_dfsi_score_column = "DFSI"
published_dfsi_rank_column = None
index_or_unnamed_columns = [column for column in dfsi.columns if column.casefold().startswith("unnamed")]
district_identifier_columns = ["Unnamed: 0"]
state_identifier_columns = ["State_Name"]

print(f"Published DFSI score column: {published_dfsi_score_column}")
print(f"Published DFSI rank column: {published_dfsi_rank_column}")
print(f"Index/unnamed CSV columns: {index_or_unnamed_columns}")
print(f"District identifier columns: {district_identifier_columns}")
print(f"State identifier columns: {state_identifier_columns}")

,column_name,stored_dtype,role,dtype,minimum,maximum,mean,median,missing_count,unique_values,integer_like,monotonic_increasing,monotonic_decreasing,resembles,identified_as_published_dfsi_score,identified_as_published_dfsi_rank,identified_as_index_or_unnamed,identified_as_district_identifier,identified_as_state_identifier,audit_note
0,Unnamed: 0,object,index_or_unnamed_csv_column; district identifier values,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False,True,True,False,"Unnamed CSV column stores district names, not a numeric rank."
1,State_Name,object,state identifier column,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False,False,False,True,State identifier column.
2,DFSI,float64,published DFSI score column,float64,4.244485,19.300564,13.588707,14.147612,1.0,743.0,False,False,True,score,True,False,False,False,False,"Only stored numeric target-like column; values resemble continuous DFSI scores, not ranks."


Published DFSI score column: DFSI
Published DFSI rank column: None
Index/unnamed CSV columns: ['Unnamed: 0']
District identifier columns: ['Unnamed: 0']
State identifier columns: ['State_Name']


## Rebuild One-to-One Reproduction Base

In [4]:
def make_review_key(value):
    if pd.isna(value):
        return pd.NA
    return re.sub(r"\s+", " ", str(value).strip()).casefold()


def split_list_cell(value):
    if pd.isna(value):
        return []
    return [part.strip() for part in str(value).split(",")]


def clean_lgd_code(value):
    if pd.isna(value):
        return pd.NA
    text = str(value).strip()
    if text == "" or text.casefold() in {"none", "nan", "na", "<na>"}:
        return pd.NA
    return text

In [5]:
event_rows = []
for _, row in flood_inventory.reset_index(names="source_row_id").iterrows():
    districts = split_list_cell(row["Districts"])
    district_codes = split_list_cell(row["District_LGD_Codes"])
    for position, district_name in enumerate(districts):
        event_rows.append({
            "source_row_id": row["source_row_id"],
            "UEI": row.get("UEI"),
            "district_name_from_inventory": district_name,
            "district_review_key": make_review_key(district_name),
            "district_lgd_code": clean_lgd_code(district_codes[position] if position < len(district_codes) else pd.NA),
        })

event_districts = pd.DataFrame(event_rows)
event_counts_by_name = (
    event_districts.groupby("district_review_key", dropna=False)
    .agg(
        flood_events=("UEI", "nunique"),
        inventory_district_names=("district_name_from_inventory", lambda s: " | ".join(sorted(set(map(str, s.dropna()))))),
        lgd_codes_seen=("district_lgd_code", lambda s: " | ".join(sorted(set(map(str, s.dropna()))))),
        lgd_code_count=("district_lgd_code", lambda s: int(s.dropna().nunique())),
    )
    .reset_index()
)

impact = flood_impact.reset_index(names="impact_source_row_id").copy()
impact["district_review_key"] = impact["Dist_Name"].map(make_review_key)
area = flooded_area.reset_index(names="area_source_row_id").copy()
area["district_review_key"] = area["Dist_Name"].map(make_review_key)
published = dfsi.reset_index(names="dfsi_source_row_id").rename(columns={"Unnamed: 0": "DFSI_Dist_Name"}).copy()
published["district_review_key"] = published["DFSI_Dist_Name"].map(make_review_key)
published["state_review_key"] = published["State_Name"].map(make_review_key)
published["published_dfsi_score"] = pd.to_numeric(published["DFSI"], errors="coerce").astype("float64")
published["published_csv_order_rank"] = published["dfsi_source_row_id"] + 1
published["published_dfsi_score_rank"] = published["published_dfsi_score"].rank(ascending=False, method="min").astype("Int64")

impact_name_counts = impact["district_review_key"].value_counts(dropna=False).rename("impact_name_count")
area_name_counts = area["district_review_key"].value_counts(dropna=False).rename("area_name_count")
published_name_counts = published["district_review_key"].value_counts(dropna=False).rename("published_name_count")

all_keys = pd.Index([])
for key_series in [impact["district_review_key"], area["district_review_key"], published["district_review_key"], event_counts_by_name["district_review_key"]]:
    all_keys = all_keys.union(pd.Index(key_series.dropna().unique()))
key_audit = pd.DataFrame({"district_review_key": all_keys})
key_audit = key_audit.merge(impact_name_counts.reset_index().rename(columns={"index": "district_review_key"}), on="district_review_key", how="left")
key_audit = key_audit.merge(area_name_counts.reset_index().rename(columns={"index": "district_review_key"}), on="district_review_key", how="left")
key_audit = key_audit.merge(published_name_counts.reset_index().rename(columns={"index": "district_review_key"}), on="district_review_key", how="left")
for column in ["impact_name_count", "area_name_count", "published_name_count"]:
    key_audit[column] = key_audit[column].fillna(0).astype(int)

one_to_one_keys = key_audit.query("impact_name_count == 1 and area_name_count == 1 and published_name_count == 1")["district_review_key"]

base = (
    impact[impact["district_review_key"].isin(one_to_one_keys)]
    .merge(area[area["district_review_key"].isin(one_to_one_keys)], on="district_review_key", how="inner", suffixes=("_impact", "_area"), validate="one_to_one")
    .merge(event_counts_by_name[["district_review_key", "flood_events", "inventory_district_names", "lgd_codes_seen", "lgd_code_count"]], on="district_review_key", how="left", validate="one_to_one")
    .merge(published[published["district_review_key"].isin(one_to_one_keys)][[
        "dfsi_source_row_id", "DFSI_Dist_Name", "State_Name", "state_review_key", "published_dfsi_score", "published_csv_order_rank", "published_dfsi_score_rank", "district_review_key"
    ]], on="district_review_key", how="inner", validate="one_to_one")
)
base["flood_events_missing_from_inventory"] = base["flood_events"].isna()
base["flood_events"] = base["flood_events"].fillna(0)

print(f"One-to-one base rows: {len(base):,}")

One-to-one base rows: 720


## Reconstruct DFSI Across Flooded-Area Interpretations

The requested equation is used here: population enters as `log10(population)`, not `log10(1 + population)`. Flooded-area variants are compared but not selected by best correlation.

In [6]:
component_source_columns = {
    "flood_events": "count distinct UEI after exploding India_Flood_Inventory_v3.Districts",
    "mean_flood_duration": "District_FloodImpact.Mean_Flood_Duration",
    "fatalities": "District_FloodImpact.Human_fatality",
    "injured": "District_FloodImpact.Human_injured",
    "population": "District_FloodImpact.Population",
}

area_variants = {
    "corrected_percent_as_released": {
        "column": "Corrected_Percent_Flooded_Area",
        "series": base["Corrected_Percent_Flooded_Area"],
        "description": "Corrected_Percent_Flooded_Area as stored in District_FloodedArea.csv",
        "paper_support": "Supported: paper describes subtracting permanent water and calculating percentage flooded area with respect to district area.",
    },
    "corrected_fraction_divide_100": {
        "column": "Corrected_Percent_Flooded_Area / 100",
        "series": base["Corrected_Percent_Flooded_Area"] / 100.0,
        "description": "Corrected_Percent_Flooded_Area divided by 100",
        "paper_support": "Useful diagnostic, but less directly supported because the released column name is a percent field.",
    },
    "percent_flooded_area_as_released": {
        "column": "Percent_Flooded_Area",
        "series": base["Percent_Flooded_Area"],
        "description": "Percent_Flooded_Area as stored in District_FloodedArea.csv",
        "paper_support": "Less supported for DFSI because paper describes subtracting permanent water before creating the flood layer.",
    },
    "percent_flooded_area_divide_100": {
        "column": "Percent_Flooded_Area / 100",
        "series": base["Percent_Flooded_Area"] / 100.0,
        "description": "Percent_Flooded_Area divided by 100",
        "paper_support": "Diagnostic only; neither corrected for permanent water nor directly aligned with the percent column scale.",
    },
}

comparisons = []
correlation_records = []

for variant_name, variant in area_variants.items():
    working = base.copy()
    working["flooded_area_variant"] = variant_name
    working["flooded_area_variant_description"] = variant["description"]
    working["flooded_area_paper_support"] = variant["paper_support"]

    working["component_flood_events"] = pd.to_numeric(working["flood_events"], errors="coerce").astype("float64")
    working["component_mean_flood_duration"] = pd.to_numeric(working["Mean_Flood_Duration"], errors="coerce").astype("float64")
    working["component_fatalities"] = pd.to_numeric(working["Human_fatality"], errors="coerce").astype("float64")
    working["component_injured"] = pd.to_numeric(working["Human_injured"], errors="coerce").astype("float64")
    working["component_flooded_area"] = pd.to_numeric(variant["series"], errors="coerce").astype("float64")
    working["component_population"] = pd.to_numeric(working["Population"], errors="coerce").astype("float64")

    component_columns = [
        "component_flood_events",
        "component_mean_flood_duration",
        "component_fatalities",
        "component_injured",
        "component_flooded_area",
        "component_population",
    ]
    for column in component_columns:
        assert working[column].dropna().ge(0).all(), f"Negative values found in {column} for {variant_name}"

    complete_mask = working[component_columns].notna().all(axis=1)
    product = (
        (1.0 + working["component_flood_events"])
        / (
            (1.0 + working["component_mean_flood_duration"])
            * (1.0 + working["component_fatalities"]) ** 2.0
            * (1.0 + working["component_injured"])
            * (1.0 + working["component_flooded_area"])
            * working["component_population"]
        )
    )
    assert product.loc[complete_mask].gt(0).all(), f"Complete-row product must be positive for {variant_name}"

    working["reconstructed_dfsi"] = (
        np.log10(1.0 + working["component_flood_events"])
        - np.log10(1.0 + working["component_mean_flood_duration"])
        - 2.0 * np.log10(1.0 + working["component_fatalities"])
        - np.log10(1.0 + working["component_injured"])
        - np.log10(1.0 + working["component_flooded_area"])
        - np.log10(working["component_population"])
    )
    working["reconstructed_dfsi_from_product"] = np.log10(product)
    working["logspace_product_abs_diff"] = (working["reconstructed_dfsi"] - working["reconstructed_dfsi_from_product"]).abs()
    working["absolute_difference"] = (working["reconstructed_dfsi"] - working["published_dfsi_score"]).abs()
    working["reconstructed_range_matches_published_approximately"] = (
        working["reconstructed_dfsi"].between(working["published_dfsi_score"].min() - 5, working["published_dfsi_score"].max() + 5)
    )
    comparisons.append(working)

    finite = working.replace([np.inf, -np.inf], np.nan).dropna(subset=["reconstructed_dfsi", "published_dfsi_score"])
    for target_name, target_series in {
        "published_dfsi_score": finite["published_dfsi_score"],
        "published_dfsi_score_rank": finite["published_dfsi_score_rank"].astype("float64"),
        "negative_published_dfsi_score_rank": -finite["published_dfsi_score_rank"].astype("float64"),
        "published_csv_order_rank": finite["published_csv_order_rank"].astype("float64"),
        "negative_published_csv_order_rank": -finite["published_csv_order_rank"].astype("float64"),
    }.items():
        correlation_records.append({
            "flooded_area_variant": variant_name,
            "flooded_area_column_or_transform": variant["column"],
            "flooded_area_paper_support": variant["paper_support"],
            "target": target_name,
            "finite_rows": len(finite),
            "pearson_correlation": finite["reconstructed_dfsi"].corr(target_series, method="pearson") if len(finite) >= 2 else np.nan,
            "spearman_correlation": finite["reconstructed_dfsi"].corr(target_series, method="spearman") if len(finite) >= 2 else np.nan,
            "mean_absolute_difference_vs_score": finite["absolute_difference"].mean() if target_name == "published_dfsi_score" and len(finite) else np.nan,
            "median_absolute_difference_vs_score": finite["absolute_difference"].median() if target_name == "published_dfsi_score" and len(finite) else np.nan,
            "maximum_absolute_difference_vs_score": finite["absolute_difference"].max() if target_name == "published_dfsi_score" and len(finite) else np.nan,
            "reconstructed_min": finite["reconstructed_dfsi"].min() if len(finite) else np.nan,
            "reconstructed_max": finite["reconstructed_dfsi"].max() if len(finite) else np.nan,
            "published_score_min": finite["published_dfsi_score"].min() if len(finite) else np.nan,
            "published_score_max": finite["published_dfsi_score"].max() if len(finite) else np.nan,
        })

all_comparisons = pd.concat(comparisons, ignore_index=True)
dfsi_all_numeric_correlations = pd.DataFrame(correlation_records)
display(dfsi_all_numeric_correlations)
dfsi_all_numeric_correlations.to_csv(TABLE_DIR / "dfsi_all_numeric_correlations.csv", index=False)

,flooded_area_variant,flooded_area_column_or_transform,flooded_area_paper_support,target,finite_rows,pearson_correlation,spearman_correlation,mean_absolute_difference_vs_score,median_absolute_difference_vs_score,maximum_absolute_difference_vs_score,reconstructed_min,reconstructed_max,published_score_min,published_score_max
0,corrected_percent_as_released,Corrected_Percent_Flooded_Area,Supported: paper describes subtracting permanent water and calculating percentage flooded area with respect to district area.,published_dfsi_score,709,-0.934496,-0.934474,23.636519,24.403468,34.351952,-15.581415,-4.406860,4.244485,19.300564
1,corrected_percent_as_released,Corrected_Percent_Flooded_Area,Supported: paper describes subtracting permanent water and calculating percentage flooded area with respect to district area.,published_dfsi_score_rank,709,0.919995,0.934474,NaN,NaN,NaN,-15.581415,-4.406860,4.244485,19.300564
2,corrected_percent_as_released,Corrected_Percent_Flooded_Area,Supported: paper describes subtracting permanent water and calculating percentage flooded area with respect to district area.,negative_published_dfsi_score_rank,709,-0.919995,-0.934474,NaN,NaN,NaN,-15.581415,-4.406860,4.244485,19.300564
3,corrected_percent_as_released,Corrected_Percent_Flooded_Area,Supported: paper describes subtracting permanent water and calculating percentage flooded area with respect to district area.,published_csv_order_rank,709,0.919995,0.934474,NaN,NaN,NaN,-15.581415,-4.406860,4.244485,19.300564
4,corrected_percent_as_released,Corrected_Percent_Flooded_Area,Supported: paper describes subtracting permanent water and calculating percentage flooded area with respect to district area.,negative_published_csv_order_rank,709,-0.919995,-0.934474,NaN,NaN,NaN,-15.581415,-4.406860,4.244485,19.300564
5,corrected_fraction_divide_100,Corrected_Percent_Flooded_Area / 100,"Useful diagnostic, but less directly supported because the released column name is a percent field.",published_dfsi_score,709,-0.904304,-0.897743,23.222892,23.955844,33.310887,-14.639827,-4.045803,4.244485,19.300564
6,corrected_fraction_divide_100,Corrected_Percent_Flooded_Area / 100,"Useful diagnostic, but less directly supported because the released column name is a percent field.",published_dfsi_score_rank,709,0.885306,0.897743,NaN,NaN,NaN,-14.639827,-4.045803,4.244485,19.300564
7,corrected_fraction_divide_100,Corrected_Percent_Flooded_Area / 100,"Useful diagnostic, but less directly supported because the released column name is a percent field.",negative_published_dfsi_score_rank,709,-0.885306,-0.897743,NaN,NaN,NaN,-14.639827,-4.045803,4.244485,19.300564
8,corrected_fraction_divide_100,Corrected_Percent_Flooded_Area / 100,"Useful diagnostic, but less directly supported because the released column name is a percent field.",published_csv_order_rank,709,0.885306,0.897743,NaN,NaN,NaN,-14.639827,-4.045803,4.244485,19.300564
9,corrected_fraction_divide_100,Corrected_Percent_Flooded_Area / 100,"Useful diagnostic, but less directly supported because the released column name is a percent field.",negative_published_csv_order_rank,709,-0.885306,-0.897743,NaN,NaN,NaN,-14.639827,-4.045803,4.244485,19.300564


## Rank Checks

In [7]:
published_rank_check = published.sort_values("published_csv_order_rank").copy()
published_rank_check["csv_order_next_score"] = published_rank_check["published_dfsi_score"].shift(-1)
published_rank_check["csv_order_nonincreasing_to_next"] = published_rank_check["published_dfsi_score"].ge(published_rank_check["csv_order_next_score"]) | published_rank_check["csv_order_next_score"].isna()
rank_1_is_largest_score = bool(published_rank_check.loc[published_rank_check["published_csv_order_rank"] == 1, "published_dfsi_score"].iloc[0] == published_rank_check["published_dfsi_score"].max())
csv_order_scores_nonincreasing = bool(published_rank_check["csv_order_nonincreasing_to_next"].all())
score_rank_is_monotone_by_definition = bool(published.dropna(subset=["published_dfsi_score"]).sort_values("published_dfsi_score_rank")["published_dfsi_score"].is_monotonic_decreasing)

print(f"CSV-order rank 1 corresponds to largest published DFSI score: {rank_1_is_largest_score}")
print(f"CSV-order rank numbers increase strictly/non-strictly as published DFSI decreases: {csv_order_scores_nonincreasing}")
print(f"Score-derived rank numbers increase as published DFSI decreases: {score_rank_is_monotone_by_definition}")

rank_inversions = published_rank_check.loc[~published_rank_check["csv_order_nonincreasing_to_next"], [
    "published_csv_order_rank", "DFSI_Dist_Name", "State_Name", "published_dfsi_score", "csv_order_next_score"
]]
display(rank_inversions.head(20))

CSV-order rank 1 corresponds to largest published DFSI score: True
CSV-order rank numbers increase strictly/non-strictly as published DFSI decreases: True
Score-derived rank numbers increase as published DFSI decreases: True


,published_csv_order_rank,DFSI_Dist_Name,State_Name,published_dfsi_score,csv_order_next_score


## Verified Comparison Using Paper/Data-Supported Flooded Area

In [8]:
supported_variant = "corrected_percent_as_released"
verified_comparison = all_comparisons.query("flooded_area_variant == @supported_variant").copy()
verified_comparison = verified_comparison[[
    "State_Name",
    "DFSI_Dist_Name",
    "Dist_Name_impact",
    "published_dfsi_score",
    "published_dfsi_score_rank",
    "published_csv_order_rank",
    "reconstructed_dfsi",
    "reconstructed_dfsi_from_product",
    "logspace_product_abs_diff",
    "absolute_difference",
    "component_flood_events",
    "component_mean_flood_duration",
    "component_fatalities",
    "component_injured",
    "component_flooded_area",
    "Percent_Flooded_Area",
    "Parmanent_Water",
    "Corrected_Percent_Flooded_Area",
    "component_population",
    "flooded_area_variant",
    "flooded_area_paper_support",
    "flood_events_missing_from_inventory",
    "lgd_codes_seen",
    "lgd_code_count",
]].rename(columns={"DFSI_Dist_Name": "district", "Dist_Name_impact": "impact_district_name"})

verified_finite = verified_comparison.replace([np.inf, -np.inf], np.nan).dropna(subset=["reconstructed_dfsi", "published_dfsi_score"])
verified_metrics = pd.DataFrame([
    {"metric": "one_to_one_comparison_rows", "value": len(verified_comparison)},
    {"metric": "one_to_one_valid_districts", "value": len(verified_finite)},
    {"metric": "missing_component_rows_preserved", "value": int(verified_comparison["reconstructed_dfsi"].isna().sum())},
    {"metric": "published_rank_column_present", "value": False},
    {"metric": "rank_1_corresponds_to_largest_published_dfsi_score_csv_order", "value": rank_1_is_largest_score},
    {"metric": "csv_order_rank_numbers_increase_as_published_dfsi_decreases", "value": csv_order_scores_nonincreasing},
    {"metric": "score_derived_rank_numbers_increase_as_published_dfsi_decreases", "value": score_rank_is_monotone_by_definition},
    {"metric": "spearman_reconstructed_vs_published_dfsi_score", "value": verified_finite["reconstructed_dfsi"].corr(verified_finite["published_dfsi_score"], method="spearman")},
    {"metric": "spearman_reconstructed_vs_published_dfsi_rank", "value": verified_finite["reconstructed_dfsi"].corr(verified_finite["published_dfsi_score_rank"].astype("float64"), method="spearman")},
    {"metric": "spearman_reconstructed_vs_negative_published_dfsi_rank", "value": verified_finite["reconstructed_dfsi"].corr(-verified_finite["published_dfsi_score_rank"].astype("float64"), method="spearman")},
    {"metric": "pearson_reconstructed_vs_published_dfsi_score", "value": verified_finite["reconstructed_dfsi"].corr(verified_finite["published_dfsi_score"], method="pearson")},
    {"metric": "mean_absolute_difference", "value": verified_finite["absolute_difference"].mean()},
    {"metric": "median_absolute_difference", "value": verified_finite["absolute_difference"].median()},
    {"metric": "maximum_absolute_difference", "value": verified_finite["absolute_difference"].max()},
    {"metric": "reconstructed_dfsi_min", "value": verified_finite["reconstructed_dfsi"].min()},
    {"metric": "reconstructed_dfsi_max", "value": verified_finite["reconstructed_dfsi"].max()},
    {"metric": "published_dfsi_score_min", "value": verified_finite["published_dfsi_score"].min()},
    {"metric": "published_dfsi_score_max", "value": verified_finite["published_dfsi_score"].max()},
    {"metric": "supported_flooded_area_interpretation", "value": supported_variant},
])

first_20 = verified_comparison.sort_values("published_csv_order_rank")[[
    "State_Name",
    "district",
    "published_dfsi_score",
    "published_dfsi_score_rank",
    "published_csv_order_rank",
    "reconstructed_dfsi",
]].head(20)

display(first_20)
display(verified_metrics)

verified_comparison.to_csv(TABLE_DIR / "dfsi_verified_comparison.csv", index=False)
verified_metrics.to_csv(TABLE_DIR / "dfsi_verified_reproduction_metrics.csv", index=False)

largest_verified_discrepancies = verified_comparison.dropna(subset=["absolute_difference"]).sort_values("absolute_difference", ascending=False).head(30)
display(largest_verified_discrepancies)

,State_Name,district,published_dfsi_score,published_dfsi_score_rank,published_csv_order_rank,reconstructed_dfsi
104,BIHAR,Patna,19.300564,1,1,-14.162612
665,WEST BENGAL,Murshidabad,18.910884,2,2,-13.984772
389,MAHARASHTRA,Thane,18.861789,3,3,-13.517729
10,ANDHRA PRADESH,Guntur,18.832069,4,4,-13.753882
671,WEST BENGAL,North Twenty Four Pargan*,18.770538,5,5,-15.581415
394,MAHARASHTRA,Nagpur,18.687114,6,6,-12.792082
676,WEST BENGAL,Purba Medinipur,18.370426,7,7,-13.657092
47,ASSAM,Lakhimpur,18.337466,8,8,-12.180345
605,UTTAR PRADESH,Gorakhpur,18.302576,9,9,-13.285187
78,BIHAR,Purba Champaran,18.295788,10,10,-14.974973


,metric,value
0,one_to_one_comparison_rows,720
1,one_to_one_valid_districts,709
2,missing_component_rows_preserved,11
3,published_rank_column_present,False
4,rank_1_corresponds_to_largest_published_dfsi_score_csv_order,True
5,csv_order_rank_numbers_increase_as_published_dfsi_decreases,True
6,score_derived_rank_numbers_increase_as_published_dfsi_decreases,True
7,spearman_reconstructed_vs_published_dfsi_score,-0.934474
8,spearman_reconstructed_vs_published_dfsi_rank,0.934474
9,spearman_reconstructed_vs_negative_published_dfsi_rank,-0.934474


,State_Name,district,impact_district_name,published_dfsi_score,published_dfsi_score_rank,published_csv_order_rank,reconstructed_dfsi,reconstructed_dfsi_from_product,logspace_product_abs_diff,absolute_difference,component_flood_events,component_mean_flood_duration,component_fatalities,component_injured,component_flooded_area,Percent_Flooded_Area,Parmanent_Water,Corrected_Percent_Flooded_Area,component_population,flooded_area_variant,flooded_area_paper_support,flood_events_missing_from_inventory,lgd_codes_seen,lgd_code_count
671,WEST BENGAL,North Twenty Four Pargan*,North Twenty Four Pargan*,18.770538,5,5,-15.581415,-15.581415,0.000000e+00,34.351952,0.0,19.0,294.0,16.0,11.225591,18.664847,7.439256,11.225591,10544399.0,corrected_percent_as_released,Supported: paper describes subtracting permanent water and calculating percentage flooded area with respect to district area.,True,NaN,NaN
104,BIHAR,Patna,Patna,19.300564,1,1,-14.162612,-14.162612,0.000000e+00,33.463177,71.0,16.0,381.0,30.0,18.820619,20.269812,1.449193,18.820619,6868950.0,corrected_percent_as_released,Supported: paper describes subtracting permanent water and calculating percentage flooded area with respect to district area.,False,520,1.0
78,BIHAR,Purba Champaran,Purba Champaran,18.295788,10,10,-14.974973,-14.974973,0.000000e+00,33.270761,0.0,19.0,184.0,20.0,9.387441,9.508604,0.121163,9.387441,6322257.0,corrected_percent_as_released,Supported: paper describes subtracting permanent water and calculating percentage flooded area with respect to district area.,True,NaN,NaN
661,WEST BENGAL,Darjiling,Darjiling,18.018981,24,24,-15.096713,-15.096713,1.776357e-15,33.115695,0.0,11.0,548.0,63.0,1.918361,2.014600,0.096239,1.918361,1849559.0,corrected_percent_as_released,Supported: paper describes subtracting permanent water and calculating percentage flooded area with respect to district area.,True,NaN,NaN
665,WEST BENGAL,Murshidabad,Murshidabad,18.910884,2,2,-13.984772,-13.984772,1.776357e-15,32.895655,44.0,23.0,435.0,9.0,10.404534,11.638689,1.234154,10.404534,8350690.0,corrected_percent_as_released,Supported: paper describes subtracting permanent water and calculating percentage flooded area with respect to district area.,False,319,1.0
159,GUJARAT,Ahmadabad,Ahmadabad,18.117122,18,18,-14.644006,-14.644006,1.776357e-15,32.761129,0.0,12.0,177.0,23.0,4.099835,4.222293,0.122458,4.099835,8738876.0,corrected_percent_as_released,Supported: paper describes subtracting permanent water and calculating percentage flooded area with respect to district area.,True,NaN,NaN
10,ANDHRA PRADESH,Guntur,Guntur,18.832069,4,4,-13.753882,-13.753882,1.776357e-15,32.585951,33.0,8.0,188.0,108.0,9.242779,10.242307,0.999528,9.242779,5374652.0,corrected_percent_as_released,Supported: paper describes subtracting permanent water and calculating percentage flooded area with respect to district area.,False,506,1.0
673,WEST BENGAL,Medinipur West,Medinipur West,17.869046,29,29,-14.708726,-14.708726,0.000000e+00,32.577772,0.0,10.0,294.0,10.0,7.812548,7.838526,0.025978,7.812548,5510554.0,corrected_percent_as_released,Supported: paper describes subtracting permanent water and calculating percentage flooded area with respect to district area.,True,NaN,NaN
389,MAHARASHTRA,Thane,Thane,18.861789,3,3,-13.517729,-13.517729,1.776357e-15,32.379518,97.0,3.0,418.0,204.0,1.253628,3.515191,2.261563,1.253628,9950177.0,corrected_percent_as_released,Supported: paper describes subtracting permanent water and calculating percentage flooded area with respect to district area.,False,497,1.0
664,WEST BENGAL,Maldah,Maldah,18.258130,14,14,-13.893744,-13.893744,0.000000e+00,32.151874,17.0,11.0,221.0,53.0,9.357036,11.776244,2.419208,9.357036,4260890.0,corrected_percent_as_released,Supported: paper describes subtracting permanent water and calculating percentage flooded area with respect to district area.,False,,0.0


## Required Answers

In [9]:
score_spearman = verified_metrics.loc[verified_metrics["metric"] == "spearman_reconstructed_vs_published_dfsi_score", "value"].iloc[0]
rank_spearman = verified_metrics.loc[verified_metrics["metric"] == "spearman_reconstructed_vs_published_dfsi_rank", "value"].iloc[0]
negative_rank_spearman = verified_metrics.loc[verified_metrics["metric"] == "spearman_reconstructed_vs_negative_published_dfsi_rank", "value"].iloc[0]
median_difference = verified_metrics.loc[verified_metrics["metric"] == "median_absolute_difference", "value"].iloc[0]
valid_districts = verified_metrics.loc[verified_metrics["metric"] == "one_to_one_valid_districts", "value"].iloc[0]

answer_summary = pd.DataFrame([
    {
        "question": "A. Was the negative correlation caused by using the rank column?",
        "answer": f"No stored rank column exists in DFSI.csv. The reconstructed DFSI is negatively correlated with the published DFSI score ({score_spearman}) and positively correlated with the score-derived rank ({rank_spearman}); this indicates the requested signed equation is directionally opposite to the published score, not that the stored target was a rank column.",
    },
    {
        "question": "B. What is the correlation with the actual published DFSI score?",
        "answer": str(score_spearman),
    },
    {
        "question": "C. What is the median numerical difference?",
        "answer": str(median_difference),
    },
    {
        "question": "D. Which flooded-area interpretation matches the published methodology?",
        "answer": "Corrected_Percent_Flooded_Area as released: the paper/data definitions describe subtracting permanent water and calculating percentage flooded area by district area.",
    },
    {
        "question": "E. How many one-to-one valid districts remain?",
        "answer": str(int(valid_districts)),
    },
])

display(answer_summary)
print("Spearman vs rank:", rank_spearman)
print("Spearman vs negative rank:", negative_rank_spearman)

,question,answer
0,A. Was the negative correlation caused by using the rank column?,No stored rank column exists in DFSI.csv. The reconstructed DFSI is negatively correlated with the published DFSI score (-0.9344741971467349) and positively correlated with the...
1,B. What is the correlation with the actual published DFSI score?,-0.9344741971467349
2,C. What is the median numerical difference?,24.403468172428695
3,D. Which flooded-area interpretation matches the published methodology?,Corrected_Percent_Flooded_Area as released: the paper/data definitions describe subtracting permanent water and calculating percentage flooded area by district area.
4,E. How many one-to-one valid districts remain?,709


Spearman vs rank: 0.9344741971467349
Spearman vs negative rank: -0.9344741971467349
